In [23]:
import camelot
import pandas as pd
from pathlib import Path 
import re

file = Path("C:\\Projects\\Python\\wealthsimple-portfolio-analytics-v2\\Data\\Data_Files\\2025-02.pdf")
tables = camelot.read_pdf(file,pages="all",flavor="stream")

matched_tables = []

for table in tables:
    df = table.df
    
    if df.astype(str).stack().str.contains("Activity - Current period", na=False).any():
        matched_tables.append(table)

for table in matched_tables:
    print(table)
    


Cannot set gray non-stroke color because /'p0' is an invalid float value
Cannot set gray non-stroke color because /'p1' is an invalid float value
Cannot set gray non-stroke color because /'p2' is an invalid float value
Cannot set gray non-stroke color because /'p3' is an invalid float value
Cannot set gray non-stroke color because /'p4' is an invalid float value
Cannot set gray non-stroke color because /'p5' is an invalid float value
Cannot set gray non-stroke color because /'p6' is an invalid float value
Cannot set gray non-stroke color because /'p7' is an invalid float value
Cannot set gray non-stroke color because /'p8' is an invalid float value
c:\Projects\Python\wealthsimple-portfolio-analytics-v2\.wsvenv\Lib\site-packages\camelot\parsers\base.py:238: UserWarning: No tables found in table area (41.35433074999997, 152.1480562499999, 554.8393307499996, 834.44805625)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)
c:\Projects\Python\wealthsimple-portfolio-an

<Table shape=(14, 5)>
<Table shape=(22, 6)>


In [ ]:
data = tables[3].df

data = data.drop([0,1])
# data.to_csv("test2.csv")

# data[0] = data[0].replace('', None)

data = data.replace(r'^\s*$', None, regex=True)

# data['crit'] = data[0].notna()
data['crit'] = data[0].notna().cumsum()
# data = data[data['crit'] != 0].copy()

data

In [18]:
df = data.copy()
max_col = df.count().idxmax()

print(df.count())
print(max_col)

def merge_text(x):
    # print(x)
    return " ".join(x.dropna().astype(str).str.strip())

agg_dict = {
    col: (merge_text if i == max_col else "first")
    for i, col in enumerate(data.columns)
    if col != "crit"
}


data = data.groupby("crit", as_index=False).agg(agg_dict).drop(columns="crit").reset_index(drop=True)

data


0        7
1       11
2        7
3        7
4        7
crit    12
dtype: int64
crit


,0,1,2,3,4
0,2025-02-03 NRT,Non-resident tax (executed at 2025-02-03),$0.25,$0.00,$88.52
1,2025-02-03 DIV,"T - AT&T, Inc.: Cash dividend distribution, re...",$0.00,$1.68,$88.52
2,2025-02-04\nBUY,CDZ - iShares S&P/TSX Canadian Dividend ETF: B...,$0.11,$0.00,$88.41
3,2025-02-04\nBUY,"T - AT&T, Inc.: Bought 0.0394 shares (executed...",$1.43,$0.00,$86.98
4,2025-02-04 DIV,ZEB - BMO Equal Weight Banks Index ETF: Cash d...,$0.00,$0.28,$87.26
5,2025-02-06\nBUY,ZEB - BMO Equal Weight Banks Index ETF: Bought...,$0.28,$0.00,$86.98
6,2025-02-10 DIV,None,$0.00,$0.19,$87.17


In [19]:
data["merged"] = data.iloc[:, :-3].astype(str).agg(" ".join, axis=1)
# data['merged'].to_csv("test3.csv")


In [ ]:
import re
import pandas as pd

# --- Regex pattern ---
pattern = r"""
^(?P<date>\d{4}-\d{2}-\d{2})\s+
(?P<transaction>[A-Z]+)

(?:\s+None)?                                    

(?:\s+(?P<ticker_id>[A-Z0-9.]+)\s*-)?

(?:.*?
    (?:
        (?:Bought|Sold)\s+(?P<quantity_buy>\d+(?:\.\d+)?)\s+shares
        |
        (?P<quantity_loan>\d+(?:\.\d+)?)\s+Shares?\s+on\s+loan
        |
        Loan\s+of\s+(?P<quantity_recall>\d+(?:\.\d+)?)\s+shares\s+terminated
    )
)?

(?:.*?\(executed\s+at\s+(?P<execDate>\d{4}-\d{2}-\d{2})\))?
"""

# --- Extract directly into data ---
extracted = data["merged"].str.extract(pattern, flags=re.VERBOSE)

# --- Combine quantity columns ---
data["quantity"] = (
    extracted["quantity_buy"]
    .fillna(extracted["quantity_loan"])
    .fillna(extracted["quantity_recall"])
)

# --- Assign final fields ---
data["date"] = extracted["date"]
data["transaction"] = extracted["transaction"]
data["ticker_id"] = extracted["ticker_id"]
data["execDate"] = extracted["execDate"]

# --- Optional: convert quantity to numeric ---
data["quantity"] = pd.to_numeric(data["quantity"], errors="coerce")

# --- Final selection (if you only want these columns) ---
# data = data[["date", "transaction", "ticker_id", "quantity", "execDate"]]

data.head(35)

,0,1,2,3,4,merged,quantity,date,transaction,ticker_id,execDate
0,2025-02-03 NRT,Non-resident tax (executed at 2025-02-03),$0.25,$0.00,$88.52,2025-02-03 NRT Non-resident tax (executed at 2...,NaN,2025-02-03,NRT,NaN,2025-02-03
1,2025-02-03 DIV,"T - AT&T, Inc.: Cash dividend distribution, re...",$0.00,$1.68,$88.52,"2025-02-03 DIV T - AT&T, Inc.: Cash dividend d...",NaN,2025-02-03,DIV,T,NaN
2,2025-02-04\nBUY,CDZ - iShares S&P/TSX Canadian Dividend ETF: B...,$0.11,$0.00,$88.41,2025-02-04\nBUY CDZ - iShares S&P/TSX Canadian...,0.0031,2025-02-04,BUY,CDZ,NaN
3,2025-02-04\nBUY,"T - AT&T, Inc.: Bought 0.0394 shares (executed...",$1.43,$0.00,$86.98,"2025-02-04\nBUY T - AT&T, Inc.: Bought 0.0394 ...",0.0394,2025-02-04,BUY,T,2025-02-03
4,2025-02-04 DIV,ZEB - BMO Equal Weight Banks Index ETF: Cash d...,$0.00,$0.28,$87.26,2025-02-04 DIV ZEB - BMO Equal Weight Banks In...,NaN,2025-02-04,DIV,ZEB,NaN
5,2025-02-06\nBUY,ZEB - BMO Equal Weight Banks Index ETF: Bought...,$0.28,$0.00,$86.98,2025-02-06\nBUY ZEB - BMO Equal Weight Banks I...,0.0067,2025-02-06,BUY,ZEB,NaN
6,2025-02-10 DIV,None,$0.00,$0.19,$87.17,2025-02-10 DIV None,NaN,2025-02-10,DIV,NaN,NaN
